# TensorFlow Decision Forest (Random Forest analogue)
Implements the tree-based baseline using [TensorFlow Decision Forests](https://www.tensorflow.org/decision_forests).

## Why tree ensembles here?
- Capture non-linear interactions between objectives, gold, and vision that linear models miss.
- Robust to unscaled features and naturally highlight feature importance for the report.
- TensorFlow Decision Forests keeps everything in a TF ecosystem.

### Hyperparameters worth tuning
| Hyperparameter | Impact | Suggested sweeps |
| --- | --- | --- |
| `num_trees` | Reduces variance; more trees = better but slower. | 200, 400, 800 |
| `max_depth` | Controls tree complexity; prevents overfitting. | None, 12, 16 |
| `min_examples` | Equivalent to `min_samples_leaf` in scikit-learn; stabilizes leaves. | 1, 2, 5 |
| `sampling_ratio` | Stochasticity per tree; default 1.0 for bagging, <1 for subsampling. | 0.7 – 1.0 |
| `categorical_algorithm` | Oblique splits vs. CART; keep `CART` for interpretability. | CART |


In [1]:
from pathlib import Path
import sys

import pandas as pd
import tensorflow_decision_forests as tfdf
from sklearn.model_selection import train_test_split

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src').exists():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError('Could not locate repo root containing src/')
    REPO_ROOT = REPO_ROOT.parent

print(f'Using repo root: {REPO_ROOT}')
DATA_PATH = REPO_ROOT / 'data/raw/high_diamond_ranked_10min.csv'
RAW = pd.read_csv(DATA_PATH)
TARGET = 'blueWins'
FEATURES = [col for col in RAW.columns if col not in {TARGET, 'gameId'}]

train_df, test_df = train_test_split(
    RAW[FEATURES + [TARGET]],
    test_size=0.2,
    stratify=RAW[TARGET],
    random_state=42,
)

train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(train_df, label=TARGET)
test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(test_df, label=TARGET)


Using repo root: /Users/liamsandy/ML_Project


### Load engineered features (reuse across models)

In [31]:
import pandas as pd

def safe_ratio(numerator, denominator, fill_value=0.0):
    denominator = denominator.replace(0, pd.NA)
    return (numerator / denominator).fillna(fill_value)

def engineer_features_inline(df):
    features = df.copy()
    blue_obj = features['blueDragons'] + features['blueHeralds'] + features['blueEliteMonsters']
    red_obj = features['redDragons'] + features['redHeralds'] + features['redEliteMonsters']
    total_obj = blue_obj + red_obj
    features['blue_objective_share'] = safe_ratio(blue_obj, total_obj, 0.5)
    features['red_objective_share'] = safe_ratio(red_obj, total_obj, 0.5)
    total_wards = features['blueWardsPlaced'] + features['redWardsPlaced']
    features['blue_vision_share'] = safe_ratio(features['blueWardsPlaced'], total_wards, 0.5)
    features['red_vision_share'] = safe_ratio(features['redWardsPlaced'], total_wards, 0.5)
    features['blue_ward_efficiency'] = safe_ratio(features['blueWardsDestroyed'], features['blueWardsPlaced'], 0.0)
    features['red_ward_efficiency'] = safe_ratio(features['redWardsDestroyed'], features['redWardsPlaced'], 0.0)
    total_kills = features['blueKills'] + features['redKills']
    features['blue_kill_share'] = safe_ratio(features['blueKills'], total_kills, 0.5)
    features['red_kill_share'] = safe_ratio(features['redKills'], total_kills, 0.5)
    features['blue_gold_per_kill'] = safe_ratio(features['blueTotalGold'], features['blueKills'] + 1)
    features['red_gold_per_kill'] = safe_ratio(features['redTotalGold'], features['redKills'] + 1)
    features['blue_xp_per_min'] = features['blueTotalExperience'] / 10.0
    features['red_xp_per_min'] = features['redTotalExperience'] / 10.0
    features['gold_obj_momentum'] = features['blueGoldDiff'] * (features['blueDragons'] + features['blueHeralds'])
    features['xp_kill_momentum'] = features['blueExperienceDiff'] * features['blueKills']
    features['gold_diff_per_min'] = features['blueGoldDiff'] / 10.0
    features['xp_diff_per_min'] = features['blueExperienceDiff'] / 10.0
    if 'gameId' in features:
        features = features.drop(columns=['gameId'])
    return features

engineered_csv = (REPO_ROOT / 'data/processed/engineered_features.csv').resolve()
if engineered_csv.exists():
    engineered_df = pd.read_csv(engineered_csv)
    print('Loaded engineered CSV:', engineered_df.shape)
else:
    print('Engineered CSV missing; computing inline.')
    engineered_df = engineer_features_inline(RAW)
    print('Inline engineered shape:', engineered_df.shape)


Loaded engineered CSV: (9879, 55)


In [33]:
from pathlib import Path
import pandas as pd

ENGINEERED_PATH = (REPO_ROOT / 'data/processed/engineered_features.csv').resolve()
if ENGINEERED_PATH.exists():
    engineered_df = pd.read_csv(ENGINEERED_PATH)
else:
    print('Engineered CSV not found; falling back to on-the-fly features.')
    from feature_engineering_playbook import engineer_features  # adjust if needed
    engineered_df = engineer_features(BASE_DF)
print('Engineered shape:', engineered_df.shape)


Engineered shape: (9879, 55)


## Build and train the forest
Set `task=tfdf.keras.Task.CLASSIFICATION` for binary outcomes.

In [9]:
rf_model = tfdf.keras.RandomForestModel(
    task=tfdf.keras.Task.CLASSIFICATION,
    num_trees=400,
    max_depth=None,
    min_examples=2,
)
rf_model.compile(metrics=['accuracy'])
rf_model.fit(x=train_ds)


Use /var/folders/pm/yqgm6225143cb3gk0n20lpr00000gn/T/tmpydkkwksr as temporary training directory
Reading training dataset...
Training dataset read in 0:00:00.346897. Found 7903 examples.
Training model...


I0000 00:00:1764460139.802506 2906705 kernel.cc:782] Start Yggdrasil model training
I0000 00:00:1764460139.802518 2906705 kernel.cc:783] Collect training examples
I0000 00:00:1764460139.802522 2906705 kernel.cc:795] Dataspec guide:
column_guides {
  column_name_pattern: "^__LABEL$"
  type: CATEGORICAL
  categorial {
    min_vocab_frequency: 0
    max_vocab_count: -1
  }
}
default_column_guide {
  categorial {
    max_vocab_count: 2000
  }
  discretized_numerical {
    maximum_num_bins: 255
  }
}
ignore_columns_without_guides: false
detect_numerical_as_discretized_numerical: false

I0000 00:00:1764460139.802594 2906705 kernel.cc:401] Number of batches: 8
I0000 00:00:1764460139.802599 2906705 kernel.cc:402] Number of examples: 7903
I0000 00:00:1764460139.804795 2906705 kernel.cc:802] Training dataset:
Number of records: 7903
Number of columns: 39

Number of columns by type:
	NUMERICAL: 38 (97.4359%)
	CATEGORICAL: 1 (2.5641%)

Columns:

NUMERICAL: 38 (97.4359%)
	1: "blueAssists" NUMERICAL

Model trained in 0:00:02.399592
Compiling model...
Model compiled.


I0000 00:00:1764460142.155179 2906705 decision_forest.cc:808] Model loaded with 400 root(s), 534862 node(s), and 38 input feature(s).
I0000 00:00:1764460142.155206 2906705 abstract_model.cc:1439] Engine "RandomForestOptPred" built
2025-11-29 18:49:02.155216: I tensorflow_decision_forests/tensorflow/ops/inference/kernel.cc:1035] Use fast generic engine


In [11]:
evaluation = rf_model.evaluate(test_ds, return_dict=True)
print(evaluation)


2/2 [==============================] - 0s 40ms/step - loss: 0.0000e+00 - accuracy: 0.7212
{'loss': 0.0, 'accuracy': 0.7211538553237915}


### Feature importance discussion
Use `rf_model.make_inspector().variable_importances()` to summarize early objectives/gold contributions.

## Engineered feature experiment (TF-DF Random Forest)

In [43]:

eng_train_df, eng_test_df = train_test_split(
    engineered_df,
    test_size=0.2,
    stratify=engineered_df[TARGET],
    random_state=42,
)
eng_train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(eng_train_df, label=TARGET)
eng_test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(eng_test_df, label=TARGET)
rf_eng = tfdf.keras.RandomForestModel(
    task=tfdf.keras.Task.CLASSIFICATION,
    num_trees=400,
    min_examples=2,
)
rf_eng.compile(metrics=['accuracy'])
rf_eng.fit(eng_train_ds)
eng_eval = rf_eng.evaluate(eng_test_ds, return_dict=True)
print('Engineered RF metrics:', eng_eval)


Use /var/folders/pm/yqgm6225143cb3gk0n20lpr00000gn/T/tmpp7zfk12n as temporary training directory
Reading training dataset...
Training dataset read in 0:00:01.140225. Found 7903 examples.
Training model...


I0000 00:00:1764463820.938062 2906705 kernel.cc:782] Start Yggdrasil model training
I0000 00:00:1764463820.938075 2906705 kernel.cc:783] Collect training examples
I0000 00:00:1764463820.938079 2906705 kernel.cc:795] Dataspec guide:
column_guides {
  column_name_pattern: "^__LABEL$"
  type: CATEGORICAL
  categorial {
    min_vocab_frequency: 0
    max_vocab_count: -1
  }
}
default_column_guide {
  categorial {
    max_vocab_count: 2000
  }
  discretized_numerical {
    maximum_num_bins: 255
  }
}
ignore_columns_without_guides: false
detect_numerical_as_discretized_numerical: false

I0000 00:00:1764463820.938172 2906705 kernel.cc:401] Number of batches: 8
I0000 00:00:1764463820.938177 2906705 kernel.cc:402] Number of examples: 7903
I0000 00:00:1764463820.941398 2906705 kernel.cc:802] Training dataset:
Number of records: 7903
Number of columns: 55

Number of columns by type:
	NUMERICAL: 54 (98.1818%)
	CATEGORICAL: 1 (1.81818%)

Columns:

NUMERICAL: 54 (98.1818%)
	1: "blueAssists" NUMERICA

Model trained in 0:00:02.631822
Compiling model...


I0000 00:00:1764463823.524234 2906705 decision_forest.cc:808] Model loaded with 400 root(s), 509200 node(s), and 54 input feature(s).
I0000 00:00:1764463823.524262 2906705 abstract_model.cc:1439] Engine "RandomForestOptPred" built
2025-11-29 19:50:23.524273: I tensorflow_decision_forests/tensorflow/ops/inference/kernel.cc:1035] Use fast generic engine


Model compiled.
2/2 [==============================] - 0s 39ms/step - loss: 0.0000e+00 - accuracy: 0.7191
Engineered RF metrics: {'loss': 0.0, 'accuracy': 0.7191295623779297}


In [41]:

print(eng_eval)

{'loss': 0.0}


In [37]:
inspector = rf_model.make_inspector()
for importance in inspector.variable_importances():
    print(importance)


SUM_SCORE
INV_MEAN_MIN_DEPTH
NUM_NODES
NUM_AS_ROOT


In [19]:
for feature in inspector.variable_importances()["INV_MEAN_MIN_DEPTH"][:10]:

SyntaxError: incomplete input (799507022.py, line 1)

In [39]:
# Summarize feature importances using several TF-DF criteria
inspector = rf_model.make_inspector()

def show_top(importances, title, k=10):
      print(f"\nTop {k} features by {title}:")
      for name, score in importances[:k]:
          print(f"  {name:<35} importance={score:.4f}")
importance_dict = inspector.variable_importances()
show_top(importance_dict["NUM_AS_ROOT"], "NUM_AS_ROOT (shows favored splitroots)")
show_top(importance_dict["INV_MEAN_MIN_DEPTH"], "INV_MEAN_MIN_DEPTH (higher = shallower splits)")
show_top(importance_dict["SUM_SCORE"], "SUM_SCORE (overall gain across tree)")

  # Quick look at metrics vs. logistic baseline
oob = inspector.evaluation()
print(f"\nTF-DF OOB Accuracy: {oob.accuracy:.3f}, LogLoss: {oob.loss:.3f}")
print("Compare with logistic regression metrics (≈0.716 acc / 0.806 ROC-AUC). Forest is only slightly higher because the dataset is mostly linear with gold/XP diff; non-linear splits add small gains but model complexity limitsimprovement.")


Top 10 features by NUM_AS_ROOT (shows favored splitroots):


TypeError: unsupported format string passed to SimpleColumnSpec.__format__